# Food Allergen Filter - Evaluation Pipeline
## Accuracy Testing & Metrics Calculation

This notebook runs the allergen detection system against test data and measures:
- Accuracy
- Precision & Recall
- F1-Score
- Confusion Matrix
- Detail Report

In [1]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from math import exp
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    model_kwargs={
        "response_logprobs": True,
        "logprobs": 1
    }
)

In [ ]:
# Load test data from JSON
with open('test_data.json', 'r', encoding='utf-8') as f:
    test_data = json.load(f)

print(f"โหลด Test Cases: {len(test_data['test_cases'])} cases")
print(f"Test cases แรก 3 ข้อ:")
for i in range(min(3, len(test_data['test_cases']))):
    case = test_data['test_cases'][i]
    print(f"\n  Case {case['id']}: {case['menu_data']['menu_name']}")
    print(f"  Ground Truth: {case['ground_truth']}")
    print(f"  Reason: {case['reason']}")

In [ ]:
def check_allergy_system(menu_data, allergy_details):
    """ตรวจสอบความเสี่ยงจากสารก่อภูมิแพ้"""
    menu_name = menu_data.get("menu_name", "")
    menu_description = menu_data.get("menu_description", "")

    allergy_names = [
        item.get("allergy_name", "").strip()
        for item in allergy_details
        if item.get("allergy_name")
    ]

    prompt = f"""ตรวจสอบเมนูอาหารว่ามีความเสี่ยงต่อสารก่อภูมิแพ้หรือไม่

menu_name: {menu_name}
menu_description: {menu_description}
allergy_names: {json.dumps(allergy_names, ensure_ascii=False)}

ให้ตอบเป็น JSON เท่านั้นในรูปแบบนี้:
{{
  "is_risky": true หรือ false,
  "matched_allergy_names": ["ชื่อสารก่อภูมิแพ้ที่พบ"],
  "self_check": {{
    "is_consistent": true หรือ false,
    "reason": "เหตุผลสั้นๆ กระชับ"
  }}
}}"""

    response = llm.invoke([HumanMessage(content=prompt)])

    try:
        parsed = json.loads(response.content)
    except json.JSONDecodeError:
        parsed = {
            "is_risky": False,
            "matched_allergy_names": [],
            "self_check": {"is_consistent": False, "reason": "parse JSON ไม่สำเร็จ"},
        }

    try:
        logprob_data = response.response_metadata.get("logprobs", {}).get("content", [])
        if logprob_data:
            chosen_logprob = logprob_data[0]["logprob"]
            confidence = exp(chosen_logprob)
        else:
            confidence = -1.0
    except (KeyError, IndexError, TypeError):
        confidence = -1.0

    matched_names = [name.strip() for name in parsed.get("matched_allergy_names", []) if isinstance(name, str)]

    matched_allergies = [
        {
            "allergy_id": item.get("allergy_id"),
            "allergy_name": item.get("allergy_name"),
            "count": item.get("count", 1),
        }
        for item in allergy_details
        if item.get("allergy_name") in matched_names
    ]

    if 0.1 <= confidence <= 0.8:
        status = "MANUAL_REVIEW"
    elif bool(parsed.get("is_risky")):
        status = "ALLERGY_WARN"
    else:
        status = "SAFE"

    self_check = parsed.get("self_check", {}) if isinstance(parsed, dict) else {}
    reason = ""
    if isinstance(self_check, dict):
        reason = str(self_check.get("reason", "")).strip()

    return {
        "menu_name": menu_name,
        "menu_description": menu_description,
        "matched_allergies": matched_allergies,
        "confidence": f"{confidence:.2%}" if confidence >= 0 else "N/A",
        "status": status,
        "is_risky": bool(parsed.get("is_risky")),
        "raw_llm": response.content,
        "reason": reason,
    }

print("Function check_allergy_system ready")

In [ ]:
# === Configuration ===
# คุณสามารถเลือก mode:
# - 'sample': รัน 10 test cases แรก (ประหยัด API)
# - 'full': รันทั้งหมด
# - 'range': ระบุช่วง เช่น (0, 20) สำหรับ 20 cases

RUN_MODE = 'sample'  # เปลี่ยนเป็น 'full' เมื่อพร้อม

if RUN_MODE == 'sample':
    test_cases = test_data['test_cases'][:10]
elif RUN_MODE == 'full':
    test_cases = test_data['test_cases']
else:
    start, end = 0, 20  # ระบุช่วงที่ต้องการ
    test_cases = test_data['test_cases'][start:end]

print(f"Mode: {RUN_MODE}")
print(f"จำนวน Test Cases ที่จะรัน: {len(test_cases)}")

In [ ]:
# === Run Evaluation ===
results = []

print("\n🔄 กำลังรันการทดสอบ...\n")

for idx, test_case in enumerate(test_cases, 1):
    case_id = test_case['id']
    menu_data = test_case['menu_data']
    allergy_details = test_case['allergy_details']
    ground_truth = test_case['ground_truth']
    
    print(f"[{idx}/{len(test_cases)}] Case {case_id}: {menu_data['menu_name'][:30]}...", end="")
    
    try:
        ai_result = check_allergy_system(menu_data, allergy_details)
        ai_prediction = ai_result['is_risky']
        
        # Mark correctness
        is_correct = (ai_prediction == ground_truth)
        
        results.append({
            'case_id': case_id,
            'menu_name': menu_data['menu_name'],
            'allergies': ', '.join([a['allergy_name'] for a in allergy_details]),
            'ground_truth': ground_truth,
            'ai_prediction': ai_prediction,
            'confidence': ai_result['confidence'],
            'status': ai_result['status'],
            'is_correct': is_correct,
            'reason': test_case['reason']
        })
        
        status_symbol = "✓" if is_correct else "✗"
        print(f" {status_symbol} [GT: {ground_truth} → AI: {ai_prediction}] Conf: {ai_result['confidence']}")
        
    except Exception as e:
        print(f" ❌ ERROR: {str(e)[:50]}")
        results.append({
            'case_id': case_id,
            'menu_name': menu_data['menu_name'],
            'allergies': ', '.join([a['allergy_name'] for a in allergy_details]),
            'ground_truth': ground_truth,
            'ai_prediction': None,
            'confidence': 'ERROR',
            'status': 'ERROR',
            'is_correct': False,
            'reason': str(e)[:100]
        })

print("\n✅ ทำการทดสอบเสร็จแล้ว")

In [ ]:
# === Convert to DataFrame for analysis ===
results_df = pd.DataFrame(results)

# Filter valid results (no errors)
valid_results = results_df[results_df['is_correct'].notna()]

print("\n" + "="*70)
print("📊 EVALUATION RESULTS")
print("="*70)

# Ground truth and predictions
y_true = valid_results['ground_truth'].values
y_pred = valid_results['ai_prediction'].values

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"\n📈 Key Metrics:")
print(f"  • Accuracy  (ถูกทั้งหมด):     {accuracy:.2%}")
print(f"  • Precision (ถูกเมื่อบอกเสี่ยง): {precision:.2%}")
print(f"  • Recall    (เจอเสี่ยงได้ทั้งหมด): {recall:.2%}")
print(f"  • F1-Score  (ความสมดุล):      {f1:.2%}")

# Count results
correct = valid_results['is_correct'].sum()
total = len(valid_results)
print(f"\n✅ ถูก: {correct}/{total} ({correct/total:.2%})")
print(f"❌ ผิด: {total - correct}/{total} ({(total-correct)/total:.2%})")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print(f"\n🔍 Confusion Matrix:")
print(f"          Predicted False  Predicted True")
print(f"Actual False    {cm[0][0]:3d}           {cm[0][1]:3d}")
print(f"Actual True     {cm[1][0]:3d}           {cm[1][1]:3d}")

In [ ]:
# === Detailed Classification Report ===
print("\n" + "="*70)
print("📋 CLASSIFICATION REPORT (ละเอียด)")
print("="*70)
print(classification_report(y_true, y_pred, target_names=['SAFE (False)', 'RISKY (True)'], digits=3))

In [ ]:
# === Show Error Cases ===
error_cases = valid_results[~valid_results['is_correct']]

print("\n" + "="*70)
print(f"❌ INCORRECT CASES: {len(error_cases)} cases")
print("="*70)

if len(error_cases) > 0:
    for idx, (_, row) in enumerate(error_cases.iterrows(), 1):
        print(f"\n{idx}. {row['menu_name']}")
        print(f"   Allergy: {row['allergies']}")
        print(f"   Ground Truth: {row['ground_truth']} → AI Predicted: {row['ai_prediction']}")
        print(f"   Confidence: {row['confidence']}")
        print(f"   Expected Reason: {row['reason']}")
else:
    print("\n✨ ไม่มีข้อผิดพลาด! AI ตอบถูกทุกข้อ")

In [ ]:
# === Visualization: Confusion Matrix ===
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['False', 'True'],
            yticklabels=['False', 'True'],
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Food Allergen Filter', fontsize=14, fontweight='bold')
plt.ylabel('Actual Ground Truth', fontsize=12)
plt.xlabel('AI Prediction', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Confusion Matrix saved to confusion_matrix.png")

In [ ]:
# === Visualization: Metrics Bar Chart ===
metrics_data = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1
}

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(metrics_data.keys(), metrics_data.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2%}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Performance Metrics - Food Allergen Filter', fontsize=14, fontweight='bold')
ax.axhline(y=0.8, color='red', linestyle='--', linewidth=1, label='80% Threshold')
ax.legend()
plt.tight_layout()
plt.savefig('metrics_chart.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Metrics Chart saved to metrics_chart.png")

In [ ]:
# === Summary: All Generated Charts ===
print("\n" + "="*70)
print("📊 ALL CHARTS GENERATED:")
print("="*70)
print("""
✅ 1. confusion_matrix.png
   → Shows TP/FP/TN/FN breakdown
   
✅ 2. metrics_chart.png
   → Accuracy, Precision, Recall, F1-Score comparison
   
✅ 3. result_breakdown.png
   → Pie chart of Correct vs Incorrect %
   
✅ 4. status_distribution.png
   → Count of SAFE / ALLERGY_WARN / MANUAL_REVIEW
   
✅ 5. accuracy_per_allergy.png
   → Accuracy for each allergy type tested
   
✅ 6. confidence_distribution.png
   → Histogram + Box plot of confidence scores
""")
print("\n💾 Files saved in the current working directory")
print("")

In [ ]:
# === Visualization: Confidence Score Distribution ===
# Extract numeric confidence values
confidence_values = []
for conf_str in valid_results['confidence']:
    try:
        # Remove % and convert to float
        if conf_str != 'N/A' and conf_str != 'ERROR':
            conf_num = float(conf_str.rstrip('%')) / 100
            confidence_values.append(conf_num)
    except:
        pass

if confidence_values:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(confidence_values, bins=15, color='#1f77b4', edgecolor='black', alpha=0.7)
    axes[0].axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='Manual Review Threshold')
    axes[0].set_xlabel('Confidence Score', fontsize=11)
    axes[0].set_ylabel('Frequency', fontsize=11)
    axes[0].set_title('Confidence Score Distribution', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Box plot by correctness
    correct_conf = [float(conf.rstrip('%'))/100 for conf, correct in zip(valid_results['confidence'], valid_results['is_correct']) 
                   if conf != 'N/A' and conf != 'ERROR' and correct]
    incorrect_conf = [float(conf.rstrip('%'))/100 for conf, correct in zip(valid_results['confidence'], valid_results['is_correct']) 
                     if conf != 'N/A' and conf != 'ERROR' and not correct]
    
    box_data = [correct_conf, incorrect_conf]
    bp = axes[1].boxplot(box_data, labels=['Correct', 'Incorrect'], patch_artist=True)
    
    for patch, color in zip(bp['boxes'], ['#2ca02c', '#d62728']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    axes[1].set_ylabel('Confidence Score', fontsize=11)
    axes[1].set_title('Confidence by Result Type', fontsize=12, fontweight='bold')
    axes[1].grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('confidence_distribution.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✅ Confidence Distribution chart saved to confidence_distribution.png")
else:
    print("⚠️  No confidence data available for visualization")

In [ ]:
# === Visualization: Accuracy Per Allergy Type ===
allergy_accuracy = {}

for allergy in valid_results['allergies'].unique():
    allergy_cases = valid_results[valid_results['allergies'] == allergy]
    if len(allergy_cases) > 0:
        acc = allergy_cases['is_correct'].sum() / len(allergy_cases)
        allergy_accuracy[allergy] = {
            'accuracy': acc,
            'count': len(allergy_cases)
        }

# Sort by accuracy
allergy_accuracy_sorted = dict(sorted(allergy_accuracy.items(), key=lambda x: x[1]['accuracy'], reverse=True))

if allergy_accuracy_sorted:
    fig, ax = plt.subplots(figsize=(12, 6))
    allergies = list(allergy_accuracy_sorted.keys())
    accuracies = [v['accuracy'] for v in allergy_accuracy_sorted.values()]
    counts = [v['count'] for v in allergy_accuracy_sorted.values()]
    
    # Color code: green for >80%, yellow for 60-80%, red for <60%
    colors = ['#2ca02c' if acc >= 0.8 else '#ff7f0e' if acc >= 0.6 else '#d62728' for acc in accuracies]
    bars = ax.bar(range(len(allergies)), accuracies, color=colors)
    
    # Add value labels and count
    for i, (bar, count) in enumerate(zip(bars, counts)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1%}\n(n={count})',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax.set_xticks(range(len(allergies)))
    ax.set_xticklabels(allergies, rotation=45, ha='right')
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_ylim(0, 1.15)
    ax.axhline(y=0.8, color='green', linestyle='--', linewidth=1.5, alpha=0.5, label='80% Threshold')
    ax.set_title('Accuracy Per Allergy Type', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('accuracy_per_allergy.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✅ Accuracy Per Allergy chart saved to accuracy_per_allergy.png")
else:
    print("⚠️  Not enough data for per-allergy analysis")

In [ ]:
# === Visualization: Status Distribution ===
status_counts = valid_results['status'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(status_counts.index, status_counts.values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])

# Add value labels
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2., f'{int(width)}',
            ha='left', va='center', fontsize=10, fontweight='bold', style='italic')

ax.set_xlabel('Count', fontsize=12)
ax.set_title('AI Status Distribution (SAFE / ALLERGY_WARN / MANUAL_REVIEW)', fontsize=13, fontweight='bold')
ax.set_xlim(0, max(status_counts.values) * 1.15)
plt.tight_layout()
plt.savefig('status_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Status Distribution chart saved to status_distribution.png")

In [ ]:
# === Visualization: Result Breakdown (Pie Chart) ===
result_counts = {
    'Correct': correct,
    'Incorrect': total - correct
}

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2ca02c', '#d62728']
wedges, texts, autotexts = ax.pie(result_counts.values(), labels=result_counts.keys(), autopct='%1.1f%%',
                                    colors=colors, startangle=90, textprops={'fontsize': 12, 'weight': 'bold'})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(11)

ax.set_title('Test Results Breakdown', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('result_breakdown.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Result Breakdown chart saved to result_breakdown.png")

In [ ]:
# === Export Results to CSV ===
results_df.to_csv('evaluation_results.csv', index=False, encoding='utf-8-sig')
print("✅ Results exported to evaluation_results.csv")

# === Export Summary Report ===
summary_report = f"""FOOD ALLERGEN FILTER - EVALUATION REPORT
{'='*70}

Run Mode: {RUN_MODE.upper()}
Test Cases Evaluated: {len(valid_results)}

OVERALL PERFORMANCE:
  Accuracy:  {accuracy:.2%}
  Precision: {precision:.2%}
  Recall:    {recall:.2%}
  F1-Score:  {f1:.2%}

RESULT BREAKDOWN:
  Correct:   {correct}/{total} ({correct/total:.2%})
  Incorrect: {total - correct}/{total} ({(total-correct)/total:.2%})

CONFUSION MATRIX:
          Predicted False  Predicted True
Actual False    {cm[0][0]:3d}           {cm[0][1]:3d}
Actual True     {cm[1][0]:3d}           {cm[1][1]:3d}

{'='*70}
Report generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

with open('evaluation_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary_report)

print("✅ Summary report saved to evaluation_summary.txt")
print("\n" + summary_report)

In [ ]:
# === Suggestions & Recommendations ===
print("\n" + "="*70)
print("💡 RECOMMENDATIONS")
print("="*70)

if accuracy >= 0.9:
    print("✨ Excellent! Accuracy >= 90%")
    print("   → Ready for production use")
elif accuracy >= 0.8:
    print("✓ Good. Accuracy >= 80%")
    print("   → Suitable for use with manual review layer")
else:
    print("⚠️  Accuracy < 80%")
    print("   → Recommend improving prompt or adding more training data")

if precision < 0.85 and recall > 0.9:
    print("\n⚠️  High False Positive Rate (too many 'RISKY' predictions)")
    print("   → Consider adjusting confidence threshold")
elif recall < 0.85 and precision > 0.9:
    print("\n⚠️  Missing Some Risk Cases (false negatives)")
    print("   → Need to improve detection sensitivity")

print("\n📌 Next Steps:")
print("   1. Review incorrect cases to identify patterns")
print("   2. Adjust prompt if patterns found")
print("   3. Increase test set to 200+ cases for full evaluation")
print("   4. Consider retraining with more examples if needed")